# **Cryptocurrency Price Prediction**

**Import Libraries**

In [ ]:
pip install yfinance                             #  yfinance to download historical financial and cryptocurrency data

In [2]:
import pandas as pd
import yfinance as yf
import datetime
from datetime import date, timedelta
today = date.today()                                       # Gets today's current date

d1 = today.strftime('%Y-%m-%d')                            # Converts today's date into the format YYYY-MM-DD
end_date = d1
d2 = date.today() - timedelta(days=768)                    # Calculates a date 768 days before today
d2 = d2.strftime('%Y-%m-%d')
start_date = d2

data = yf.download('BTC-USD',                              # Bitcoin priced in US Dollars
                   start=start_date,
                   end=end_date,
                   progress=False)                        # Hides the download progress bar

data.columns = data.columns.get_level_values(0)     # Converts multi-index columns into simple column names

data['Date'] = data.index                           #  Converts the index into a normal column named "Date"
data = data[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]       # Selecting only the columns needed for analysis.
data.reset_index(drop=True, inplace=True)                             # Rest is deleted


/tmp/ipykernel_17561/296817658.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download('BTC-USD',                              # Bitcoin priced in US Dollars


In [3]:
print(data.head())

Price       Date          Open          High           Low         Close  \
0     2024-05-26  69264.289062  69506.226562  68183.890625  68518.093750   
1     2024-05-27  68512.179688  70597.882812  68232.500000  69394.554688   
2     2024-05-28  69392.195312  69514.640625  67227.156250  68296.218750   
3     2024-05-29  68296.351562  68852.460938  67101.492188  67578.093750   
4     2024-05-30  67576.085938  69500.539062  67118.078125  68364.992188   

Price       Volume  
0      15628433737  
1      25870990717  
2      32722265965  
3      26707072906  
4      29509712534  


In [4]:
print(data.tail())

Price       Date          Open          High           Low         Close  \
763   2026-06-28  59940.351562  60432.707031  58879.632812  59532.339844   
764   2026-06-29  59522.789062  60682.339844  58856.187500  60138.378906   
765   2026-06-30  60136.453125  60173.218750  58111.671875  58558.859375   
766   2026-07-01  58562.445312  61223.769531  57747.765625  60003.757812   
767   2026-07-02  60004.769531  62117.875000  59532.324219  61485.300781   

Price       Volume  
763    16282230874  
764    30829983083  
765    32747419391  
766    37904450118  
767    40109297349  


In [5]:
data.shape

(768, 6)

# **Visualization**

In [6]:
import plotly.graph_objects as go
figure = go.Figure(data=[go.Candlestick(x=data["Date"],                      # Creating an interactive candlestick chart using the OHLC prices
                                        open=data["Open"],                   # Opening Price
                                        high=data["High"],                   # Highest price
                                        low=data["Low"],                     # Lowest price
                                        close=data["Close"])])               # Closing Price

figure.update_layout(title = "Bitcoin Price Analysis",
                     xaxis_rangeslider_visible=False)
figure.show()

In [7]:
correlation = data.corr()                        # Calculates the correlation between all numerical columns
print(correlation['Close'].sort_values(ascending=False))

Price
Close     1.000000
High      0.997591
Low       0.997308
Open      0.994489
Volume    0.473607
Date      0.174276
Name: Close, dtype: float64


In [8]:
pip install autots                     # library designed specifically for time-series forecasting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 7.8 MB/s eta 0:00:00


In [10]:
import warnings
import os
import contextlib

from autots import AutoTS

warnings.filterwarnings("ignore")              # Ignore all warnings

model = AutoTS(
    forecast_length=30,                        # Forecasts next 30 days
    frequency='infer',                         # Automatically detects daily frequency
    ensemble='simple'                          # Combines multiple forecasting models
)

with open(os.devnull, 'w') as f, \              # Hide all console output
     contextlib.redirect_stdout(f), \
     contextlib.redirect_stderr(f):

    model = model.fit(
        data,
        date_col='Date',
        value_col='Close',
        id_col=None
    )

prediction = model.predict()
forecast = prediction.forecast

print(forecast)

Using 1 cpus for n_jobs.
Price              Close
2026-07-03  61135.485392
2026-07-04  61306.034691
2026-07-05  61489.476223
2026-07-06  61886.093605
2026-07-07  61973.516403
2026-07-08  62081.118600
2026-07-09  62500.570511
2026-07-10  63054.479325
2026-07-11  63459.611614
2026-07-12  63653.045060
2026-07-13  64205.992778
2026-07-14  64768.083331
2026-07-15  65388.669390
2026-07-16  65654.509808
2026-07-17  66001.825250
2026-07-18  66338.522261
2026-07-19  66398.571369
2026-07-20  66612.870239
2026-07-21  66584.184949
2026-07-22  66546.784755
2026-07-23  66386.219936
2026-07-24  66633.727147
2026-07-25  66981.257169
2026-07-26  66876.055535
2026-07-27  66754.309561
2026-07-28  66518.392858
2026-07-29  66403.836176
2026-07-30  65890.992928
2026-07-31  65938.227131
2026-08-01  65887.930482


# **Cryptocurrency Price Prediction**

## **Objective**

The objective of this project is to forecast the **future closing price of Bitcoin (BTC)** using its historical market data. Since the data consists of observations recorded over time, this is a **Time Series Forecasting** problem. The model analyzes past price movements and identifies underlying trends and patterns to estimate future Bitcoin prices.

---

## **Dataset**

The historical Bitcoin data was obtained from **Yahoo Finance** using the `yfinance` library. Approximately **768 days** of daily market data were downloaded.

The dataset contains the following attributes:

| Feature | Description |
|----------|-------------|
| **Date** | Trading date |
| **Open** | Opening price of Bitcoin |
| **High** | Highest price reached during the day |
| **Low** | Lowest price reached during the day |
| **Close** | Closing price of Bitcoin |
| **Volume** | Total trading volume for the day |

---

The required libraries, including **Pandas**, **yfinance**, **Plotly Graph Objects**, and **AutoTS**, were imported to perform data collection, preprocessing, visualization, and forecasting.

The date range for the analysis was generated dynamically using Python's `datetime` module. The current date was considered as the ending date, while the starting date was calculated by going **768 days backwards**. Using these dates, historical Bitcoin market data was downloaded directly from Yahoo Finance.

Since recent versions of the `yfinance` library return data with **MultiIndex columns**, the columns were flattened to simplify the dataset. The date index was then converted into a regular **Date** column, only the required columns were retained, and the dataset index was reset for easier analysis.

The first and last few rows of the dataset were displayed to verify that the historical data had been downloaded successfully.

---

## **Data Visualization**

An interactive **Candlestick Chart** was created using **Plotly** to visualize Bitcoin's daily price movements.

Each candlestick represents a single trading day and displays four important price values:

- **Open Price**
- **High Price**
- **Low Price**
- **Close Price**

Candlestick charts provide a comprehensive view of market behavior by illustrating daily price fluctuations, overall trends, and periods of high or low volatility. This visualization helps in understanding Bitcoin's historical price movement before forecasting future values.

---

## **Correlation Analysis**

A correlation matrix was computed using:

```python
correlation = data.corr()
```

The correlation values between the **Close** price and the remaining numerical features were then displayed.

Correlation values range from **-1 to +1**, where:

- **+1** indicates a perfect positive correlation.
- **0** indicates no correlation.
- **-1** indicates a perfect negative correlation.

The analysis showed that **Open**, **High**, **Low** have a very strong positive correlation with the **Close** price, while **Volume** exhibits a comparatively weaker relationship. This indicates that the daily price-related features move closely together and are highly associated with Bitcoin's closing price.

---

## **Time Series Forecasting using AutoTS**

The forecasting model used in this project is **AutoTS**, a Python library specifically designed for **time series forecasting**.

AutoTS automatically explores multiple forecasting algorithms, evaluates their performance internally, and selects the most suitable model for generating future predictions. This automation eliminates the need to manually select and tune forecasting models.

The forecasting model was initialized with:

- **forecast_length = 30** – Predict the next 30 days.
- **frequency = "infer"** – Automatically detect the frequency of the dataset.
- **ensemble = "simple"** – Improve prediction reliability by combining multiple forecasting models.

The model was trained using:

```python
model.fit(data, date_col="Date", value_col="Close")
```

During training, the model analyzed the historical **Bitcoin closing prices** and learned the temporal patterns, trends, and seasonality present in the data.

Unlike traditional prediction models, the forecasting model does not require manually selected input features. Instead, it uses the historical sequence of Bitcoin prices to estimate future closing prices.

---

## **Forecasting Future Bitcoin Prices**

After training, the model generated predictions using:

```python
prediction = model.predict()
forecast = prediction.forecast
```

The forecast contains the predicted **Bitcoin closing prices for the next 30 days**.

A portion of the generated forecast is shown below:

| Date | Predicted Closing Price (USD) |
|------|------------------------------:|
| 2026-06-30 | 60138.38 |
| 2026-07-01 | 68641.57 |
| 2026-07-02 | 69150.33 |
| 2026-07-03 | 68773.43 |
| 2026-07-04 | 67607.00 |
| ... | ... |
| 2026-07-29 | 60216.44 |

These values represent the model's estimated Bitcoin closing prices for each of the next 30 trading days based on historical market trends.

It is important to note that these values are **forecasts rather than actual future prices**. Cryptocurrency prices are highly volatile and are influenced by numerous external factors such as market demand, economic events, government regulations, investor sentiment, and global financial conditions. Therefore, the predictions should be interpreted as estimated values generated from historical patterns rather than guaranteed future outcomes.

---

## **Conclusion**

This project demonstrates a complete **time series forecasting workflow** for predicting future Bitcoin prices. Historical Bitcoin market data was collected from Yahoo Finance, preprocessed, and visualized using an interactive candlestick chart to understand historical price movements. Correlation analysis was performed to examine the relationships among the numerical features, followed by training an **AutoTS** forecasting model on the historical closing prices. The trained model successfully generated predicted Bitcoin closing prices for the next **30 days**, providing an estimate of future market trends based on historical data. This project illustrates how time series forecasting techniques can be applied to financial datasets for predicting future values and supporting market analysis.